# 00 - Bản chất bài toán Kẻ mạo danh và Cấu trúc Dữ liệu

Notebook này mở đầu chuỗi bài giảng thực nghiệm cho bài toán **"Kẻ mạo danh" (The Impostor)**:
- Tìm hiểu cấu trúc một mẫu dữ liệu cặp ảnh và ràng buộc nhãn nhị phân.
- Quy tắc đọc ID dạng chuỗi (`str`) nhằm bảo toàn các chữ số `0` ở đầu.
- Phân chia tập dữ liệu: 800 cặp Development (chia 3-fold CV) và 200 cặp Holdout độc lập.
- So sánh chỉ số đánh giá: Macro-F1 vs Accuracy qua ví dụ số nhỏ minh họa.
- Khám phá trực quan một cặp ảnh thật lấy từ tập TRAIN của Fold 0.

## 1. Bản chất bài toán cặp ảnh

Trong bài toán Kẻ mạo danh, mỗi mẫu dữ liệu đầu vào gồm một cặp 2 ảnh:
- `image_0`: Ảnh ở vị trí bên trái (chỉ số 0).
- `image_1`: Ảnh ở vị trí bên phải (chỉ số 1).

Ràng buộc xác định từ đề bài là: **trong mỗi cặp luôn có đúng một ảnh thật và một ảnh giả mạo**. Nhiệm vụ của mô hình là xác định vị trí của ảnh giả:
- $\text{fake\_position} = 0$: `image_0` là ảnh giả, `image_1` là ảnh thật.
- $\text{fake\_position} = 1$: `image_1` là ảnh giả, `image_0` là ảnh thật.

Lưu ý: Chúng ta không có thông tin về cơ chế ghép cặp ban đầu của ban tổ chức (có cùng chủ thể hay điều kiện chụp hay không). Mô hình cần tập trung phát hiện các sai khác về chất lượng hoặc dấu vết số học giữa hai ảnh.

In [ ]:
from pathlib import Path
import os
import sys

# Thiết lập đường dẫn tìm thư mục KeMaoDanh cho dù chạy từ repo root hay thư mục notebooks
def find_task_root():
    cwd = Path.cwd().resolve()
    for cand in [cwd, cwd.parent, cwd / 'KeMaoDanh', cwd.parent / 'KeMaoDanh']:
        if (cand / 'src/kmd').is_dir() and (cand / 'configs').is_dir():
            return cand.resolve()
    raise FileNotFoundError("Mở notebook từ repo root, KeMaoDanh hoặc KeMaoDanh/notebooks.")

TASK_ROOT = find_task_root()
if str(TASK_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(TASK_ROOT / 'src'))

import numpy as np
import pandas as pd
from kmd.core import PACKAGE, read_csv, split_fold, metric
from kmd.pipeline import prepare_development

print(f"Thư mục gốc của bài toán: {TASK_ROOT}")

## 2. Cấu trúc thư mục và quy tắc giữ số 0 đầu ID

Dữ liệu thực tế được đặt tại thư mục:
```text
data/
├── train/
│   ├── pairs.csv           # Chứa: pair_id, image_0, image_1, fake_position
│   └── images/...          # Các file ảnh nguyên bản
└── test/                   # (Tùy chọn) Bộ dự đoán không nhãn
    ├── pairs.csv           # Chứa: pair_id, image_0, image_1
    └── images/...
```

**Lưu ý quan trọng về ID:** Cột `pair_id` có các chuỗi chứa số 0 ở đầu (ví dụ: `"00001"`, `"00295"`). Khi đọc dữ liệu bằng `pandas`, bắt buộc phải chỉ định `dtype={'pair_id': str}` (như hàm `read_csv` trong `kmd.core`) để không bị ép kiểu thành số nguyên `295`, tránh làm lệch thứ tự khi ghép kết quả.

In [ ]:
data_root_env = os.environ.get('DATA_ROOT')
data_root = Path(data_root_env or TASK_ROOT / 'data/train').expanduser().resolve()
if not (data_root / 'pairs.csv').is_file():
    raise FileNotFoundError(f'Thiếu dữ liệu train: {data_root / "pairs.csv"}. Xem README để đặt DATA_ROOT.')

print(f"Đường dẫn tập train: {data_root}")
if (data_root / 'pairs.csv').is_file():
    dev_preview = prepare_development(data_root)
    print("Kiểu ID:", dev_preview.pair_id.dtype)
    print("Ba dòng development (không hiển thị holdout):")
    print(dev_preview.head(3))
else:
    print("Chưa tìm thấy data/train/pairs.csv. Bạn có thể đặt dữ liệu vào data/train/ hoặc gán DATA_ROOT.")

## 3. Phân chia 800 Development và 200 Holdout

Tập train gốc gồm 1000 cặp ảnh. Để xây dựng mô hình một cách khoa học:
- **800 cặp Development:** Được cố định trong `configs/development_split.csv` và chia thành 3 fold cân bằng (Fold 0, Fold 1, Fold 2). Toàn bộ quá trình trích xuất đặc trưng, huấn luyện mô hình và đánh giá Out-of-Fold (OOF) chỉ thực hiện trên 800 cặp này.
- **200 cặp Holdout:** Không dùng để fit scaler, chọn phương pháp/checkpoint hoặc báo điểm trong repo này, nhằm giữ vai trò như một tập kiểm tra độc lập.

Một **fold** là một nhóm ID cố định. Ở thí nghiệm fold 0, nhóm 0 là validation để đo kết quả; nhóm 1 và 2 là train để học tham số. “Development” gồm cả ba nhóm, không đồng nghĩa với train của riêng một thí nghiệm. Loader đọc manifest gốc để kiểm tra schema rồi chỉ lấy 800 ID được phép; các notebook không xem ảnh hoặc đánh giá 200 cặp còn lại.

In [ ]:
split_path = TASK_ROOT / 'configs/development_split.csv'
split_df = read_csv(split_path)

print(f"Số lượng cặp trong development split: {len(split_df)}")
print("Phân bố số lượng cặp theo từng fold:")
print(split_df['inner_fold'].value_counts().sort_index())

## 4. Chỉ số đánh giá: Macro-F1 vs Accuracy

Đề bài sử dụng **Macro-F1** làm thước đo xếp hạng chính:
- $\text{Accuracy} = \frac{\text{Số dự đoán đúng}}{\text{Tổng số mẫu}}$.
- $\text{Macro-F1} = \frac{F1_{\text{lớp 0}} + F1_{\text{lớp 1}}}{2}$, trong đó $F1_c = \frac{2 \cdot P_c \cdot R_c}{P_c + R_c}$.

### Ví dụ minh họa bằng số nhỏ
Giả sử có 10 cặp ảnh gồm: **8 cặp lớp 0** và **2 cặp lớp 1**.
- **Mô hình A (luôn đoán lớp 0):** Đúng 8/10 mẫu $\rightarrow \text{Accuracy} = 80\%$. Tuy nhiên, với lớp 1, Precision = 0, Recall = 0 $\rightarrow F1_1 = 0$. Điểm $\text{Macro-F1} = \frac{0.889 + 0.0}{2} \approx 0.444$.
- **Mô hình B (đoán đúng 7 mẫu lớp 0 và 1 mẫu lớp 1):** $\text{Accuracy} = 80\%$, nhưng điểm $\text{Macro-F1} \approx 0.688$.

Macro-F1 phạt rất nặng việc bỏ sót hoàn toàn một lớp nhãn.

Với từng lớp c, **precision P** trả lời “trong các mẫu đoán là c, bao nhiêu mẫu đúng?”, còn **recall R** trả lời “trong các mẫu thật sự thuộc c, tìm được bao nhiêu?”. Ví dụ B có P₀=R₀=7/8 và P₁=R₁=1/2 nên Macro-F1=(0.875+0.5)/2=0.6875. Khi không có dự đoán cho một lớp, precision không xác định; code quy ước bằng 0. Các xác suất ở cell minh họa được đặt bằng tay để giải thích metric, không phải kết quả mô hình trên dữ liệu cuộc thi.

In [ ]:
# Tính toán thực tế trên ví dụ số nhỏ
y_true = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1])

# Mô hình A: Dự đoán toàn bộ là lớp 0 (xác suất p = 0.1)
p_pred_a = np.full(10, 0.1)

# Mô hình B: Dự đoán có phân biệt hai lớp
p_pred_b = np.array([0.1, 0.2, 0.1, 0.3, 0.2, 0.1, 0.4, 0.8, 0.9, 0.3])

res_a = metric(y_true, p_pred_a)
res_b = metric(y_true, p_pred_b)

print(f"Mô hình A (Đoán lệch hoàn toàn): Accuracy = {res_a['accuracy']:.3f}, Macro-F1 = {res_a['macro_f1']:.3f}")
print(f"Mô hình B (Dự đoán cả hai lớp):  Accuracy = {res_b['accuracy']:.3f}, Macro-F1 = {res_b['macro_f1']:.3f}")

## 5. Khám phá một cặp ảnh từ tập TRAIN của Fold 0

Để không gây rò rỉ thông tin từ tập validation hay holdout, ta nạp 800 cặp development, chia fold và lấy một mẫu cụ thể từ **tập TRAIN của Fold 0** để hiển thị cả hai ảnh cạnh nhau.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

if (data_root / 'pairs.csv').is_file():
    dev_frame = prepare_development(data_root)
    tr_fold0, _ = split_fold(dev_frame, fold=0)
    
    sample_row = tr_fold0.iloc[0]
    p0 = data_root / sample_row['image_0']
    p1 = data_root / sample_row['image_1']
    
    if p0.is_file() and p1.is_file():
        im0 = Image.open(p0)
        im1 = Image.open(p1)
        
        fig, axes = plt.subplots(1, 2, figsize=(11, 5))
        axes[0].imshow(im0)
        axes[0].set_title(f"image_0 ({sample_row['image_0']})\nKích thước: {im0.size} | Dung lượng: {p0.stat().st_size:,} B")
        axes[0].axis('off')
        
        axes[1].imshow(im1)
        axes[1].set_title(f"image_1 ({sample_row['image_1']})\nKích thước: {im1.size} | Dung lượng: {p1.stat().st_size:,} B")
        axes[1].axis('off')
        
        label_text = f"Ảnh giả: image_{sample_row['fake_position']}"
        fig.suptitle(f"Mẫu TRAIN Fold 0: {sample_row['pair_id']} - {label_text}", fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()
    else:
        print(f"Không tìm thấy file ảnh tại: {p0} hoặc {p1}")
else:
    print("Cần dataset tại data/train để hiển thị ảnh.")

## 6. Tổng kết và chuyển tiếp

Chúng ta đã nắm rõ bản chất bài toán phân loại nhị phân theo cặp, cấu trúc phân chia 800 Development / 200 Holdout, và thước đo Macro-F1.

Bằng mắt thường, sự khác biệt giữa hai ảnh có thể rất khó nhận diện. Trong bài tiếp theo, chúng ta sẽ trích xuất 32 đặc trưng thống kê số học và xây dựng mô hình Logistic Regression đầu tiên.